In [145]:
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix


In [146]:

# source: https://www.kaggle.com/datasets/rashikrahmanpritom/heart-attack-analysis-prediction-dataset
df = pd.read_csv("heart.csv")
df.head()

,age,sex,cp,trtbps,chol,fbs,restecg,thalachh,exng,oldpeak,slp,caa,thall,output
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [147]:
print(df.attrs.get("description", "No description provided."))

No description provided.


In [148]:
y = df['output']
X = df.drop(columns=['output'])
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trtbps    303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalachh  303 non-null    int64  
 8   exng      303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slp       303 non-null    int64  
 11  caa       303 non-null    int64  
 12  thall     303 non-null    int64  
dtypes: float64(1), int64(12)
memory usage: 30.9 KB


In [149]:
X_scaled = StandardScaler().fit(X).transform(X)

In [150]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
# Define the loss function
loss_fn = nn.BCEWithLogitsLoss()

In [152]:
# Builds a simple feedforward neural network for binary classification
# with one hidden layer and ReLU activation.
# The input layer has 12 features, the hidden layer has 12 neurons,
# and the output layer has 2 classes (0 and 1).

# ANNreg = nn.Sequential(
#     nn.Linear(13,8),  # input layer
#     nn.ReLU(),       # activation function
#     nn.Linear(8,1)   # output layer
#     )

ANNreg = nn.Sequential(
    nn.Linear(13, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

In [153]:
# convert to Tensor
X_train_tensor = torch.from_numpy(X_train).float()
y_train_tensor = torch.from_numpy(y_train.values) # numpy int64 to long
y_train_tensor.type()

'torch.LongTensor'

In [154]:
# Pro Tip. Use DataLoader for batching and shuffling

from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X_train_tensor, y_train_tensor)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [155]:
def manual_sgd(model, lr):
    for layer in model:
        if isinstance(layer, nn.Linear):
            with torch.no_grad():
                layer.weight -= lr * layer.weight.grad
                layer.bias   -= lr * layer.bias.grad

                layer.weight.grad.zero_()
                layer.bias.grad.zero_()


In [156]:
lr = 0.01  # learning rate

for epoch in range(200):  # number of epochs
    for x_batch, y_batch in loader:
        output = ANNreg(x_batch)
        y_batch = y_batch.float()
        loss = loss_fn(output.squeeze(), y_batch)
        loss.backward()

        # Manual update
        manual_sgd(ANNreg, lr)

In [157]:
for i, layer in enumerate(ANNreg):
    if isinstance(layer, nn.Linear):
        print(f"Layer {i} weights:\n", layer.weight.data)
        print(f"Layer {i} biases:\n", layer.bias.data)

Layer 0 weights:
 tensor([[-1.6289e-01,  1.2925e-02, -2.5813e-01,  3.8799e-01, -1.6347e-01,
         -5.7204e-03, -9.6741e-03, -1.8658e-01,  4.0784e-01,  2.3385e-01,
         -8.4303e-02,  6.6873e-02,  4.4413e-01],
        [-1.4385e-01, -2.5405e-01,  5.3079e-02,  5.1346e-02, -2.6145e-01,
          1.0581e-01, -1.4414e-02,  3.1935e-01, -8.9761e-02, -1.9287e-01,
         -1.4873e-01, -2.0916e-01, -1.0286e-01],
        [ 4.2889e-02, -3.1005e-02,  8.3367e-02, -8.9467e-02, -1.5054e-01,
          2.0472e-01, -2.4026e-01, -2.1583e-01, -1.7298e-01,  1.2147e-01,
         -5.6247e-02,  1.7160e-01, -1.4004e-01],
        [-2.5780e-01, -3.9096e-02,  3.9333e-05, -9.6134e-02, -4.9391e-02,
          2.3924e-01, -1.4399e-01,  8.3997e-02, -6.8570e-02,  4.0894e-02,
          6.6717e-02,  2.3168e-01,  2.6400e-01],
        [ 5.2090e-02, -2.8199e-01,  1.4347e-02,  9.3375e-02,  1.7200e-01,
         -1.0782e-01,  1.6259e-01,  2.0585e-01, -3.0003e-01, -1.5300e-01,
          3.0522e-01, -7.6639e-02, -7.7314e-02

In [161]:
# show the losses

# manually compute losses
# final forward pass
X_test_tensor = torch.from_numpy(X_test).float()

logits = ANNreg(X_test_tensor)              # raw model output
probs = torch.sigmoid(logits).squeeze()     # probabilities ∈ (0, 1)

thresholds = np.linspace(0.1, 0.9, 9)
for t in thresholds:
    y_hat = (probs > t).long()                # predicted class labels (0 or 1)
    y_true = torch.from_numpy(y_test.values).long()  # ground truth labels
    correct = (y_hat == y_true).sum().item()
    accuracy = correct / len(y_true)    
    cm = confusion_matrix(y_true.numpy(), y_hat.numpy())
    print("Confusion Matrix:\n", cm)    
    print(f"Threshold: {t:.2f} → Accuracy: {accuracy:.4f}")



Confusion Matrix:
 [[12 17]
 [ 0 32]]
Threshold: 0.10 → Accuracy: 0.7213
Confusion Matrix:
 [[23  6]
 [ 3 29]]
Threshold: 0.20 → Accuracy: 0.8525
Confusion Matrix:
 [[24  5]
 [ 3 29]]
Threshold: 0.30 → Accuracy: 0.8689
Confusion Matrix:
 [[25  4]
 [ 3 29]]
Threshold: 0.40 → Accuracy: 0.8852
Confusion Matrix:
 [[25  4]
 [ 3 29]]
Threshold: 0.50 → Accuracy: 0.8852
Confusion Matrix:
 [[25  4]
 [ 9 23]]
Threshold: 0.60 → Accuracy: 0.7869
Confusion Matrix:
 [[26  3]
 [13 19]]
Threshold: 0.70 → Accuracy: 0.7377
Confusion Matrix:
 [[27  2]
 [15 17]]
Threshold: 0.80 → Accuracy: 0.7213
Confusion Matrix:
 [[29  0]
 [22 10]]
Threshold: 0.90 → Accuracy: 0.6393


### 🎯 Threshold = `0.40`

```
[[TN=25, FP=4],
 [FN=3,  TP=29]]
```

* **False Negatives (FN)**: 3  ✅ *Low* → **You’re catching most positives**
* **False Positives (FP)**: 4
* **Implication**: You’re **erring on the side of caution** — more likely to predict heart attack if there's doubt.

---

### 🎯 Threshold = `0.50`

```
[[TN=25, FP=4],
 [FN=9,  TP=23]]
```

* **False Negatives (FN)**: 9 ❌ *Much higher*
* **False Positives (FP)**: 4
* **Implication**: You’re **missing more true positives** — less conservative

---

## 🩺 Real-World Context

In medical prediction (e.g. **heart attack risk**):

> **False Negatives are more dangerous** than False Positives
> → It's better to raise a red flag unnecessarily than to miss a real risk.

So:

### ✅ Recommended Threshold: `0.40`

* Same overall accuracy
* Much **lower FN**
* Higher **recall** (you catch more true heart attack cases)


In [43]:
# Calculate predictions manually

import torch
import torch.nn.functional as F

# Dummy input (example with shape [1, 13])
x = torch.tensor([[0.1]*13], dtype=torch.float32)  # shape (1, 13)

# Extract weights from trained model
W0 = ANNreg[0].weight.detach()    # shape (8, 13)
b0 = ANNreg[0].bias.detach()      # shape (8,)
W2 = ANNreg[2].weight.detach()    # shape (2, 8)
b2 = ANNreg[2].bias.detach()      # shape (2,)

# Step 1: Linear layer 0
h = x @ W0.T + b0         # shape (1, 8)

# Step 2: ReLU
h_relu = F.relu(h)        # shape (1, 8)

# Step 3: Linear layer 2
y = h_relu @ W2.T + b2    # shape (1, 2)

print("Manual prediction:", y)


Manual prediction: tensor([[-0.1831, -0.3810]])
